# マテリアルズ・インフォマティクス入門ハンズオン



## 環境確認

必要なパッケージが import できることを確認します。


In [ ]:
import ase
import asap3
import pymatgen
import nglview
import gpaw
import chgnet
import torch
import mp_api
import crystal_toolkit
import dash
import plotly
import pandas as pd

print("ase", ase.__version__)
print("asap3", getattr(asap3, "__version__", "ok"))
print("pymatgen", pymatgen.__version__)
print("nglview", nglview.__version__)
print("gpaw", gpaw.__version__)
print("chgnet", chgnet.__version__)
print("torch", torch.__version__, "cuda=", torch.cuda.is_available())
print("mp_api", getattr(mp_api, "__version__", "ok"))
print("crystal_toolkit", getattr(crystal_toolkit, "__version__", "ok"))
print("dash", dash.__version__)
print("plotly", plotly.__version__)
print("pandas", pd.__version__)


## 構造作成

物質を原子レベルでデータとして扱うには、原子の種類と座標の情報を格納する必要があります。

ここでは、ASEパッケージの `Atoms` オブジェクトを使うことにしましょう。

### 水素分子 $\text{H}_2$

原子種と座標を直接指定して $\text{H}_2$ を作ります。


In [ ]:
from ase import Atoms
import nglview as nv

h2 = Atoms("H2", positions=[[0.0, 0.0, 0.0], [0.74, 0.0, 0.0]])
print(h2)
print("positions:\n", h2.positions)

view_h2 = nv.show_ase(h2)
view_h2.add_ball_and_stick()
view_h2

### チャレンジ：分子を作ってみましょう

好きな分子を作ってみましょう。

In [ ]:
my_molecule = Atoms("OCO", positions=[[0.0, 0.0, 0.0], [0.9, 0.0, 0.0], [1.8, 0.0, 0.0]])
print(my_molecule)
view_mol = nv.show_ase(my_molecule)
view_mol.add_ball_and_stick()
view_mol

### 2.2 水分子 $\text{H}_2\text{O}$

ASE の `molecule` 関数で登録済み分子を簡単に作れます。


In [ ]:
from ase.build import molecule

h2o = molecule("H2O")
print(h2o)
print("symbols :", h2o.get_chemical_symbols())
print("positions:\n", h2o.positions)

view_h2o = nv.show_ase(h2o)
view_h2o.add_ball_and_stick()
view_h2o


利用可能な分子名の一部は次のように確認できます。


In [ ]:
from ase.collections import g2

print("登録分子数:", len(g2.names))
print([name for name in g2.names if name in ("H2", "H2O", "O2", "N2", "CO2", "CH4")])


### 2.3 NaCl 結晶（rocksalt）

`bulk` で岩石塩型 NaCl を作ります。`a` は従来型立方格子の格子定数です（実験値はおよそ 5.64 Å）。


In [ ]:
from ase.build import bulk

nacl = bulk("NaCl", crystalstructure="rocksalt", a=5.64, cubic=True)
print(nacl)
print("cell lengths:", nacl.cell.lengths())
print("pbc:", nacl.pbc)

# 見やすくするため 2x2x2 超格子を表示
nacl_view = nacl * (2, 2, 2)
view_nacl = nv.show_ase(nacl_view)
view_nacl.add_unitcell()
view_nacl


pymatgen でも同様の構造を作れ、ASE と相互変換できます。


In [ ]:
from pymatgen.core import Structure, Lattice
from pymatgen.io.ase import AseAtomsAdaptor

structure = Structure(
    Lattice.cubic(5.64),
    ["Na", "Cl"],
    [[0.0, 0.0, 0.0], [0.5, 0.5, 0.5]],
)
atoms_from_pmg = AseAtomsAdaptor.get_atoms(structure)
print(structure)
print(atoms_from_pmg)


## 3. Materials Project から構造を取得

[Materials Project](https://materialsproject.org/) は、DFT 計算に基づく大規模な材料データベースです。
Python クライアント `mp-api` の `MPRester` を使うと、組成・物性条件から構造を検索・取得できます。

参考: [Materials Project Workshop — Materials API](https://workshop.materialsproject.org/lessons/04_materials_api/MAPI%20Lesson%20(filled)/)、
[公式ドキュメント](https://docs.materialsproject.org/downloading-data/using-the-api/getting-started)


### 3.1 API キーの準備

1. [Materials Project](https://next-gen.materialsproject.org/) にログイン
2. 右上の **API** から API キーを取得
3. 下のセルでキーを設定するか、環境変数 `MP_API_KEY` を設定

> API キーは他人に公開しないでください。ノートブックを共有する場合はキーをセルに直書きせず、環境変数を使います。


In [ ]:
import os

# どちらか一方を設定:
# 1) 環境変数 MP_API_KEY（推奨）
# 2) 下の文字列に自分のキーを入れる（ローカル作業時のみ）
MP_API_KEY = os.environ.get("MP_API_KEY", "")

if not MP_API_KEY:
    # 例: MP_API_KEY = "your_api_key_here"
    raise ValueError(
        "MP_API_KEY が未設定です。"
        "https://next-gen.materialsproject.org/api でキーを取得し、"
        "環境変数かこのセルで設定してください。"
    )

print("API key: 設定済み（先頭4文字）=", MP_API_KEY[:4], "...")


### 3.2 material_id で構造を1件取得

Materials Project の各物質には `mp-XXXX` 形式の ID があります。
例: ダイヤモンド型 Si は `mp-149`、岩塩型 NaCl は `mp-22862`。

`get_structure_by_material_id` が最も手軽です。


In [ ]:
from mp_api.client import MPRester
import nglview as nv
from pymatgen.io.ase import AseAtomsAdaptor

with MPRester(MP_API_KEY) as mpr:
    # Si (diamond): mp-149
    si_structure = mpr.get_structure_by_material_id("mp-149")

print(si_structure)
print("formula:", si_structure.composition.reduced_formula)
print("lattice a,b,c:", si_structure.lattice.abc)

si_atoms = AseAtomsAdaptor.get_atoms(si_structure)
view_si = nv.show_ase(si_atoms)
view_si.add_unitcell()
view_si


### 3.3 組成・化学系で検索する

現在の API では、旧来の `MPRester.query(...)` の代わりに
`mpr.materials.summary.search(...)` を使います。

- `formula="NaCl"` … 組成式で検索
- `chemsys="Na-Cl"` … 化学系（元素の組み合わせ）で検索
- `fields=[...]` … 返す項目を絞ると高速
- `energy_above_hull=(0, 0.05)` … 熱力学的に安定に近いものだけ、など


In [ ]:
with MPRester(MP_API_KEY) as mpr:
    docs = mpr.materials.summary.search(
        formula="NaCl",
        fields=[
            "material_id",
            "formula_pretty",
            "structure",
            "energy_above_hull",
            "band_gap",
            "symmetry",
        ],
    )

print(f"NaCl 組成のヒット数: {len(docs)}")
for doc in docs[:5]:
    sg = doc.symmetry.symbol if doc.symmetry else "?"
    print(
        f"{doc.material_id}  {doc.formula_pretty:8s}  "
        f"E_hull={doc.energy_above_hull:.4f} eV/atom  "
        f"Eg={doc.band_gap:.3f} eV  SG={sg}"
    )


安定相（`energy_above_hull` が小さいもの）の構造を取り出し、ASE に変換して可視化します。


In [ ]:
# energy_above_hull が最小のものを採用
docs_sorted = sorted(docs, key=lambda d: d.energy_above_hull)
best = docs_sorted[0]
print("選択:", best.material_id, best.formula_pretty, f"E_hull={best.energy_above_hull:.6f}")

nacl_mp = best.structure
nacl_atoms_mp = AseAtomsAdaptor.get_atoms(nacl_mp)
print(nacl_atoms_mp)
print("cell lengths:", nacl_atoms_mp.cell.lengths())

view_nacl_mp = nv.show_ase(nacl_atoms_mp * (2, 2, 2))
view_nacl_mp.add_unitcell()
view_nacl_mp


### 3.4 化学系での検索例（Si–O）

Workshop と同様に、化学系 `Si-O` で候補を絞り、物性条件を付けてスクリーニングできます。


In [ ]:
with MPRester(MP_API_KEY) as mpr:
    sio_docs = mpr.materials.summary.search(
        chemsys="Si-O",
        energy_above_hull=(0.0, 0.05),  # 比較的安定
        fields=["material_id", "formula_pretty", "energy_above_hull", "band_gap", "nsites"],
    )

print(f"Si-O 系（E_hull ≤ 50 meV/atom）: {len(sio_docs)} 件")
for doc in sorted(sio_docs, key=lambda d: d.energy_above_hull)[:8]:
    print(
        f"{doc.material_id}  {doc.formula_pretty:10s}  "
        f"nsites={doc.nsites:3d}  "
        f"E_hull={doc.energy_above_hull:.4f}  Eg={doc.band_gap:.3f}"
    )


取得した `Structure` は以降の GPAW・CHGNet・MD にもそのまま使えます。
例えば NaCl の DFT 節では、手で `bulk(...)` する代わりに Materials Project の格子定数を初期値にできます。


## 4. GPAW による NaCl の DFT（格子定数スキャン）

第一原理計算ソフト **GPAW** を使い、格子定数 $a$ を変えながら NaCl の全エネルギーを計算します。
エネルギーが最小になる $a$ が、この計算条件での平衡格子定数の目安です。

Binder 向けにカットオフ・k点・スキャン点数を小さくしています（数分かかる場合があります）。


In [ ]:
from pathlib import Path
import japanize_matplotlib
import numpy as np
import matplotlib.pyplot as plt
from ase.build import bulk
from gpaw import GPAW, PW

Path("output").mkdir(exist_ok=True)

# 実験値 5.64 Å 付近を粗くスキャン
a_list = np.linspace(3.2, 8.0, 30)
energies = []

for a in a_list:
    atoms = bulk("NaCl", crystalstructure="rocksalt", a=a)  # primitive (2 atoms)
    atoms.calc = GPAW(
        mode=PW(200),
        xc="PBE",
        kpts=(2, 2, 2),
        txt=f"output/nacl_a{a:.2f}.txt",
    )
    e = atoms.get_potential_energy()
    energies.append(e)
    print(f"a = {a:.3f} Å -> E = {e:.4f} eV  ({e / len(atoms):.4f} eV/atom)")

energies = np.array(energies)
a_min = a_list[np.argmin(energies)]
print(f"\n最小エネルギー付近の格子定数: a ≈ {a_min:.3f} Å")


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(a_list, energies / 2.0, "o-", label="GPAW PBE")
ax.axvline(5.64, color="gray", ls="--", label="実験値 ≈ 5.64 Å")
ax.set_xlabel("格子定数 a [Å]")
ax.set_ylabel("エネルギー [eV/atom]")
ax.set_title("NaCl rocksalt: E–a 曲線 (GPAW)")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("output/nacl_Ea.png", dpi=120)
plt.show()


> **補足:** 本番計算では平面波カットオフ・k点密度を十分大きくし、収束を確認します。
> ここではハンズオン用に意図的に軽い設定にしています。


## 5. ASAP (EMT) による Al の MD

[ASAP](https://wiki.fysik.dtu.dk/asap/) の **EMT**（Effective Medium Theory）力場は、金属の高速な古典 MD に便利です。
ここでは fcc-Al の NVE（Velocity Verlet）シミュレーションを短時間実行します。

参考: [Atomistic Simulation Tutorial (Matlantis)](https://docs.matlantis.com/atomistic-simulation-tutorial/ja/index.html)


In [ ]:
import os
from time import perf_counter

from asap3 import EMT
from ase.build import bulk
from ase import units
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.md.verlet import VelocityVerlet
from ase.md import MDLogger

Path("output").mkdir(exist_ok=True)

atoms = bulk("Al", "fcc", a=4.05, cubic=True)
atoms *= (3, 3, 3)  # 108 atoms
atoms.pbc = True
atoms.calc = EMT()

time_step = 1.0       # fs
temperature = 800     # K
num_md_steps = 20000
num_interval = 100

MaxwellBoltzmannDistribution(atoms, temperature_K=temperature, force_temp=True)
Stationary(atoms)

traj_file = "output/al_asap_nve.traj"
log_file = "output/al_asap_nve.log"
for path in (traj_file, log_file):
    if os.path.exists(path):
        os.remove(path)

dyn = VelocityVerlet(
    atoms,
    time_step * units.fs,
    trajectory=traj_file,
    loginterval=num_interval,
)

temperatures = []
times_fs = []

def record():
    step = dyn.get_number_of_steps()
    temperatures.append(atoms.get_temperature())
    times_fs.append(step * time_step)
    if step % (num_interval * 5) == 0:
        print(
            f"step={step:5d}  "
            f"Etot={atoms.get_total_energy():.4f} eV  "
            f"T={atoms.get_temperature():.1f} K"
        )

dyn.attach(record, interval=num_interval)
dyn.attach(MDLogger(dyn, atoms, log_file, header=True, stress=False, peratom=False, mode="w"), interval=num_interval)

t0 = perf_counter()
print("ASAP EMT / Al NVE MD 開始")
dyn.run(num_md_steps)
print(f"完了: {perf_counter() - t0:.1f} s")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nglview as nv
from ase.io import read

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(np.array(times_fs) / 1000.0, temperatures, "-")
ax.set_xlabel("時間 [ps]")
ax.set_ylabel("温度 [K]")
ax.set_title("fcc-Al NVE (ASAP EMT)")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

traj = read(traj_file, index=":")
print(f"trajectory frames: {len(traj)}")

v = nv.show_asetraj(traj, gui=True)
v.add_unitcell()
v

NVE では総エネルギーがほぼ保存され、温度は目標値の周りで揺らぎます。
温度を厳密に制御したい場合は NVT（例: `NVTBerendsen`）を使います。


## 6. CHGNet による MD

[CHGNet](https://chgnet.lbl.gov/) は結晶の Universal Machine Learning Potential です。
DFT よりはるかに速く、多様な組成に適用できます。

ここでは Li の BCC 構造に対し、短い NVT MD を実行します。


In [ ]:
import os
from pathlib import Path
from time import perf_counter

from ase.build import bulk
from chgnet.model import CHGNet
from chgnet.model.dynamics import MolecularDynamics

Path("output").mkdir(exist_ok=True)

atoms = bulk("Li", "bcc", a=3.51, cubic=True)
atoms *= (2, 2, 2)

chgnet_model = CHGNet.load()

traj_file = "output/li_chgnet_nvt.traj"
log_file = "output/li_chgnet_nvt.log"
for path in (traj_file, log_file):
    if os.path.exists(path):
        os.remove(path)

md = MolecularDynamics(
    atoms=atoms,
    model=chgnet_model,
    ensemble="nvt",
    thermostat="Berendsen",
    temperature=300,
    timestep=2,  # fs
    trajectory=traj_file,
    logfile=log_file,
    loginterval=10,
    use_device="cpu",
)

print("CHGNet NVT MD 開始")
t0 = perf_counter()
md.run(200)  # 0.4 ps
print(f"完了: {perf_counter() - t0:.1f} s")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nglview as nv
from ase.io import read

traj = read(traj_file, index=":")
print(f"frames = {len(traj)}")
print(f"最終構造: {traj[-1]}")

with open(log_file) as f:
    header = f.readline().strip()
print("log header:", header)

log = np.loadtxt(log_file, skiprows=1)
if log.ndim == 1:
    log = log.reshape(1, -1)

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(log[:, 0], log[:, -1], "-")
ax.set_xlabel("時間 [ps]")
ax.set_ylabel("温度 [K]")
ax.set_title("Li BCC NVT (CHGNet)")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

view_li = nv.show_ase(traj[-1])
view_li.add_unitcell()
view_li


### 単点エネルギーの例

MD の前に、CHGNet Calculator で単点計算もできます。


In [ ]:
from ase.build import bulk
from chgnet.model.dynamics import CHGNetCalculator

atoms_sp = bulk("Li", "bcc", a=3.51)
atoms_sp.calc = CHGNetCalculator(model=chgnet_model, use_device="cpu")
print(f"Li BCC energy = {atoms_sp.get_potential_energy():.4f} eV")


## 7. CHGNet 構造最適化と Crystal Toolkit

参考: [crystaltoolkit_relax_viewer.ipynb](https://github.com/CederGroupHub/chgnet/blob/main/examples/crystaltoolkit_relax_viewer.ipynb)

CHGNet の `StructOptimizer` で構造緩和し、軌跡を [Crystal Toolkit](https://docs.crystaltoolkit.org/)（Dash アプリ）でインタラクティブに可視化します。
例題は Materials Project の LiMnO₂（[mp-18767](https://materialsproject.org/materials/mp-18767)）です。


### 7.1 構造の読み込みと摂動

平衡構造を少し乱し、セルも膨らませてから緩和します。


In [ ]:
import numpy as np
from pymatgen.core import Structure

structure = Structure.from_file("input/mp-18767-LiMnO2.cif")
print("original:", structure.get_space_group_info())

# 原子座標を微小摂動
rng = np.random.default_rng(0)
for site in structure:
    site.coords += rng.normal(size=3) * 0.3

# セル体積を 10% 拡大
structure.scale_lattice(structure.volume * 1.1)
print("perturbed:", structure.get_space_group_info())
print(structure)


### 7.2 StructOptimizer で緩和

`FIRE` オプティマイザで力と応力が小さくなるまで緩和します（Binder では `use_device="cpu"`）。


In [ ]:
import pandas as pd
from chgnet.model import StructOptimizer

relaxer = StructOptimizer(use_device="cpu")
result = relaxer.relax(structure, fmax=0.1, steps=200, verbose=True)
trajectory = result["trajectory"]

print("final structure:")
print(result["final_structure"])
print(f"final energy = {trajectory.energies[-1]:.4f} eV")


In [ ]:
e_col = "Energy (eV)"
force_col = "Force (eV/Å)"
df_traj = pd.DataFrame(trajectory.energies, columns=[e_col])
df_traj[force_col] = [
    np.linalg.norm(force, axis=1).mean()
    for force in trajectory.forces
]
df_traj.index.name = "step"
df_traj.tail()


### 7.3 Crystal Toolkit で緩和軌跡を可視化

スライダーでステップを動かすと、構造とエネルギー・力のグラフが連動します。


In [ ]:
import crystal_toolkit.components as ctc
import plotly.graph_objects as go
from crystal_toolkit.settings import SETTINGS
from dash import Dash, dcc, html
from dash.dependencies import Input, Output
from pymatgen.core import Structure

mp_id = "mp-18767"
dft_energy = -59.09  # eV; see https://materialsproject.org/materials/mp-18767

app = Dash(prevent_initial_callbacks=True, assets_folder=SETTINGS.ASSETS_PATH)

step_size = max(1, len(trajectory) // 20)
slider = dcc.Slider(
    id="slider",
    min=0,
    max=len(trajectory) - 1,
    step=step_size,
    updatemode="drag",
)


def plot_energy_and_forces(df, step, e_col, force_col, title):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df.index, y=df[e_col], mode="lines", name="Energy"))
    line_color = fig.data[0].line.color
    fig.add_trace(
        go.Scatter(x=df.index, y=df[force_col], mode="lines", name="Forces", yaxis="y2")
    )
    fig.update_layout(
        template="plotly_white",
        title=title,
        xaxis=dict(title="Relaxation Step"),
        yaxis=dict(title=e_col),
        yaxis2=dict(title=force_col, overlaying="y", side="right"),
        legend=dict(yanchor="top", y=1, xanchor="right", x=1),
    )
    fig.add_vline(x=step, line=dict(dash="dash", width=1))
    fig.add_hline(
        y=dft_energy,
        line=dict(dash="dot", width=1, color=line_color),
        annotation=dict(text="DFT final energy", yanchor="top"),
    )
    return fig


def make_title(spg_symbol, spg_num):
    href = f"https://materialsproject.org/materials/{mp_id}/"
    return f"<a href='{href}'>{mp_id}</a> - {spg_symbol} ({spg_num})"


# 表示用に別インスタンスを用意（コールバックで更新）
view_structure = Structure.from_file("input/mp-18767-LiMnO2.cif")
title = make_title(*view_structure.get_space_group_info())

graph = dcc.Graph(
    id="fig",
    figure=plot_energy_and_forces(df_traj, 0, e_col, force_col, title),
    style={"maxWidth": "50%"},
)
struct_comp = ctc.StructureMoleculeComponent(id="structure", struct_or_mol=view_structure)

app.layout = html.Div(
    [
        html.H1("Structure Relaxation Trajectory", style=dict(margin="1em", fontSize="2em")),
        html.P("スライダーを動かして、各緩和ステップの構造を確認してください。"),
        slider,
        html.Div([struct_comp.layout(), graph], style=dict(display="flex", gap="2em")),
    ],
    style=dict(margin="auto", textAlign="center", maxWidth="1200px", padding="2em"),
)
ctc.register_crystal_toolkit(app=app, layout=app.layout)


@app.callback(
    Output(struct_comp.id(), "data"),
    Output(graph, "figure"),
    Input(slider, "value"),
)
def update_structure(step: int):
    step = int(step or 0)
    lattice = trajectory.cells[step]
    coords = trajectory.atom_positions[step]
    view_structure.lattice = lattice
    assert len(view_structure) == len(coords)
    for site, coord in zip(view_structure, coords, strict=True):
        site.coords = coord
    title = make_title(*view_structure.get_space_group_info())
    fig = plot_energy_and_forces(df_traj, step, e_col, force_col, title)
    return view_structure, fig


app.run(height=800, use_reloader=False)


## 8. 金属表面での酸化反応（CHGNet）

参考: [Atomistic Simulation Tutorial 6.2 — 計算事例３：金属表面での酸化反応](https://docs.matlantis.com/atomistic-simulation-tutorial/ja/6_2_md-nvt.html)

キレイな fcc-Al の (111) 表面が酸素雰囲気下にさらされた状態を考えます。
Al は室温でも酸化しやすいので、NVT-MD でその様子を再現できるか試します。

モデルは **fcc-Al(111) スラブ + 真空層 10 Å + ランダム配置した O₂ 分子 20 個** です。


In [ ]:
from pathlib import Path

import nglview as nv
from ase.io import read

Path("output").mkdir(exist_ok=True)

atoms = read("input/fcc111_Al_3x4x6_vac10A_20O2.cif")
atoms.pbc = True
print(atoms)
print({s: list(atoms.symbols).count(s) for s in sorted(set(atoms.symbols))})

view0 = nv.show_ase(atoms)
view0.add_unitcell()
view0


<figure style="text-align: center">
<img src="assets/Fig6-2_fcc111_Al_3x4x6_vac10A_20O2.png" width="420"/>
<figcaption>初期構造: fcc-Al(111) スラブと 20 個の O₂（チュートリアル Fig.6-2g）</figcaption>
</figure>

元チュートリアルでは Matlantis の PFP を使っています。ここでは Binder で使える **CHGNet**（汎用 ML ポテンシャル）に置き換えます。
金属表面と気相分子の反応を扱うには、O を含む系に対応したポテンシャルが必要です（ASAP の EMT では不向きです）。

300 K・NVT（Nosé–Hoover）で短い MD を実行します。


### Nosé–Hoover 熱浴

ASE では `NPT` クラスに `pfactor=None` を渡すと、体積一定の **Nosé–Hoover NVT** になります。

- https://wiki.fysik.dtu.dk/ase/ase/md.html#nose-hoover-dynamics

時定数 `ttime`（$\tau_T$）はおよそ 20–25 fs が目安です。小さすぎると不安定、大きすぎると温度収束が遅くなります。


In [ ]:
import os
from time import perf_counter

from ase import units
from ase.md import MDLogger
from ase.md.npt import NPT
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from chgnet.model import CHGNet
from chgnet.model.dynamics import CHGNetCalculator

chgnet_model = CHGNet.load()
atoms.calc = CHGNetCalculator(model=chgnet_model, use_device="cpu")

time_step = 1.0       # fs
temperature = 300     # K
num_md_steps = 200    # Binder 向けテスト。本格計算は 10000 程度
num_interval = 50
ttime = 20.0          # thermostat time constant [fs]

traj_file = "output/al111_o2_nvt_nosehoover.traj"
log_file = "output/al111_o2_nvt_nosehoover.log"
for path in (traj_file, log_file):
    if os.path.exists(path):
        os.remove(path)

MaxwellBoltzmannDistribution(atoms, temperature_K=temperature, force_temp=True)
Stationary(atoms)

dyn = NPT(
    atoms,
    time_step * units.fs,
    temperature_K=temperature,
    externalstress=0.1e-6 * units.GPa,  # NVT では実質無視
    ttime=ttime * units.fs,
    pfactor=None,  # None → NVT (Nosé–Hoover)
    loginterval=num_interval,
    trajectory=traj_file,
)

def print_dyn():
    step = dyn.get_number_of_steps()
    print(
        f"step={step:5d}  "
        f"Etot={atoms.get_total_energy():.3f} eV  "
        f"T={atoms.get_temperature():.1f} K"
    )

dyn.attach(print_dyn, interval=num_interval)
dyn.attach(
    MDLogger(dyn, atoms, log_file, header=True, stress=True, peratom=True, mode="w"),
    interval=num_interval,
)

print("Al(111)+O2 Nosé–Hoover NVT (CHGNet) 開始")
t0 = perf_counter()
dyn.run(num_md_steps)
print(f"完了: {perf_counter() - t0:.1f} s")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read

traj = read(traj_file, index=":")
print(f"frames = {len(traj)}")

with open(log_file) as f:
    header = f.readline().strip()
print("header:", header)

log = np.loadtxt(log_file, skiprows=1)
if log.ndim == 1:
    log = log.reshape(1, -1)

# ASE MDLogger: Time[ps] ... T[K] は通常5列目 (index 4)
time_ps = log[:, 0]
temp_K = log[:, 4] if log.shape[1] > 4 else log[:, -1]

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(time_ps, temp_K, "-")
ax.set_xlabel("時間 [ps]")
ax.set_ylabel("温度 [K]")
ax.set_title("Al(111)+O₂ NVT (CHGNet / Nosé–Hoover)")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

view1 = nv.show_ase(traj[-1])
view1.add_unitcell()
view1


短いテスト計算でも、O₂ が表面近傍へ近づく様子が見えることがあります。
本格的な時間スケール（数 ps〜）では、チュートリアルと同様に **吸着 → O–O 開裂 → 表面酸化** が進みます。

アルミニウムは室温・低酸素分圧でも酸化物を作りやすいため、狭いセルに多数の O₂ を詰めた条件では酸化が起きやすいです（[Ellingham diagram](https://en.wikipedia.org/wiki/Ellingham_diagram) も参照）。

<figure style="text-align: center">
<img src="assets/Fig6-2_O2_adsorption_on_fcc111_Al.png" width="520"/>
<figcaption>O₂ の吸着・開裂プロセスの模式（チュートリアル Fig.6-2i）</figcaption>
</figure>

このように NVT-MD では、ガスと固体表面の反応ダイナミクスを原子レベルで追跡できます。


## まとめ

| 内容 | ツール | ポイント |
|------|--------|----------|
| H₂ / H₂O / NaCl の作成 | ASE (`Atoms`, `molecule`, `bulk`) | 構造はすべての計算の出発点 |
| データベースから構造取得 | Materials Project (`mp-api`) | `material_id`・組成・化学系で検索 |
| NaCl 格子定数スキャン | GPAW (DFT) | $E(a)$ から平衡格子定数を見積もる |
| Al の MD | ASAP EMT | 金属向けの高速古典力場 |
| Li の MD | CHGNet | 汎用 ML ポテンシャル |
| LiMnO₂ 構造最適化 | CHGNet `StructOptimizer` + Crystal Toolkit | 緩和軌跡のインタラクティブ可視化 |
| Al(111) 表面酸化 | CHGNet + Nosé–Hoover NVT | 吸着・解離・酸化のダイナミクス |

次のステップとしては、NVT/NPT アンサンブル、表面・欠陥モデルなどが挙げられます。
詳細は [Atomistic Simulation Tutorial](https://docs.matlantis.com/atomistic-simulation-tutorial/ja/index.html)、
[CHGNet examples](https://github.com/CederGroupHub/chgnet/tree/main/examples)、
[Materials Project Workshop](https://workshop.materialsproject.org/) を参照してください。
